# Detailed Notes for Models

## ESM-2
- **Purpose**:  
  - **What it does**: Learns general protein sequence patterns by predicting masked amino acids.  
  - **Focus**: Captures evolutionary sequence relationships for downstream tasks.  
  - **Output**: Probability distributions over masked amino acids in protein sequences.  
- **Name of the model**: ESM-2 (family of models scaled from 8 million to 15 billion parameters); ESMFold is the structure prediction extension.  
- **Output of the model**: ESM-2 outputs representations capturing sequence patterns and can predict masked amino acids (e.g., for perplexity evaluation). Specifically, for each masked position (15% of the sequence), it provides a probability distribution over the 20 standard amino acids based on the unmasked context.  
- **Training Data**: ESM-2 is trained on UniRef sequences (~65 million unique sequences seen during training from ~43 million UniRef50 clusters, derived from ~138 million UniRef90 sequences).  
- **Data Processing**: ESM-2: Random 15% masking of amino acids in sequences; no MSA required.  
- **Encoding of the Epitope/MHC**: No specific epitope/MHC encoding mentioned (unlike some immunology-focused models that might use BLOSUM32); ESM-2 operates on raw amino acid sequences.  
- **General Architecture**: ESM-2: Transformer with attention mechanisms (scaled up to 15 billion parameters).  
- **Loss functions**: ESM-2: Masked language modeling loss: $\mathcal{L}_{\text{MLM}} = \sum_{i \in M} \log p(x_i | x_{\setminus M})$, where 15% of positions are masked.

## ESMFold
- **Purpose**:  
  - **What it does**: Predicts atomic-level 3D protein structures from single sequences.  
  - **Focus**: Fast, accurate structure prediction without multiple sequence alignments.  
  - **Output**: 3D coordinates and confidence scores for protein structures.  
- **Name of the model**: ESMFold (an end-to-end structure prediction model built on ESM-2, specifically using the 3-billion-parameter ESM-2 variant in the paper’s evaluations).  
- **Output of the model**: ESMFold outputs atomic-level 3D coordinates for all atoms in a protein structure, derived from a single sequence input, along with per-residue confidence scores (predicted LDDT, pLDDT) and a global confidence metric (predicted TM-score, pTM). These outputs enable structural analysis and accuracy estimation without needing MSAs.  
- **Training Data**: ESMFold’s folding head is trained on around 325,000 experimentally determined structures from the PDB (25,000 clusters) and augmented with ~12 million high-confidence structures predicted by AlphaFold2, providing a mix of real and synthetic data for learning structural patterns.  
- **Data Processing**: ESMFold uses ESM-2’s pre-trained representations as input (no additional masking needed at this stage); the folding head processes these representations, leveraging the PDB and AlphaFold2 data for supervision. No MSA or template search is required, unlike AlphaFold2 or RoseTTAFold.  
- **Encoding of the Epitope/MHC**: No specific epitope/MHC encoding is mentioned; ESMFold inherits ESM-2’s approach of working directly with raw amino acid sequences, making it agnostic to specific immunological features.  
- **General Architecture**: ESMFold builds on ESM-2’s transformer architecture, adding a folding trunk (alternating updates to sequence and pairwise representations) and a structure module (transformer-based, with three recycling steps to refine coordinates), simplifying the pipeline compared to MSA-dependent models.  
- **Loss functions**: ESMFold is trained with losses akin to AlphaFold’s (e.g., errors between predicted and true atomic coordinates), though the paper doesn’t specify the exact formulation, instead referencing AlphaFold’s methodology (likely including frame-aligned point error and other structural loss terms).

## MHCflurry 2.0 BA
- **Purpose**:  
  - **What it does**: Predicts how strongly a peptide binds to a specific MHC class I allele.  
  - **Focus**: Allele-specific binding affinity, the most selective step in MHC presentation.  
  - **Output**: A binding affinity score (transformed to 0-1 from nM), indicating peptide-MHC interaction strength.  
- **Name of the model**: MHCflurry 2.0 BA (pan-allele binding affinity predictor, supports 14,993 MHC class I alleles).  
- **Output of the model**: Predicts binding affinity in nanomolar (nM) units for a peptide-MHC pair, transformed to a 0-1 scale (1 - log5000(nM affinity)); higher values indicate stronger binding.  
- **Training Data**: Combines 493,473 MS-identified ligands (e.g., from IEDB, SysteMHC Atlas, Sarkizova et al., Abelin et al.) and 219,596 in vitro affinity measurements (e.g., IEDB, BD2013 dataset); MS hits assigned “less than 100 nM” affinity.  
- **Data Processing**: Peptides (8-15 mers) padded to 45-mer fixed length by concatenating left-aligned, centered, and right-aligned versions with “X” placeholders; MHC alleles represented as 37-mer pseudosequences from a multi-species alignment; both encoded with BLOSUM62 matrix.  
- **Encoding**: BLOSUM62 substitution matrix transforms each amino acid (and “X”) into a 21-dimensional vector for peptides and MHC pseudosequences.  
- **MHC/Epitope Encoding**: Peptides and MHC alleles are explicitly encoded with BLOSUM62; MHC uses 37 positions (34 from NetMHCpan + 3 additional) to capture allele-specific binding preferences.  
- **Architecture**: Ensemble of 10 feedforward neural networks (from 140 candidates); each has 2-3 dense layers (256-1024 units), dropout (50%), and optional skip connections; inputs are concatenated peptide and allele encodings.  
- **Loss Functions**: Variant of mean squared error (MSE) that handles inequality labels (e.g., “less than 100 nM” for MS data), optimized with pre-training on synthetic data and early stopping.

## MHCflurry 2.0 AP
- **Purpose**:  
  - **What it does**: Predicts the likelihood a peptide is processed (e.g., cleaved by proteasomes, transported by TAP, trimmed by ERAP) before MHC binding.  
  - **Focus**: Allele-independent steps of antigen processing that occur prior to binding.  
  - **Output**: A probability score (0-1) of processing, independent of MHC allele.  
- **Name of the model**: MHCflurry 2.0 AP (allele-independent antigen processing predictor, with variants “with flanks” and “without flanks”).  
- **Output of the model**: Outputs a probability score (0-1) indicating the likelihood a peptide is processed and presented, independent of MHC allele; trained to distinguish MS hits from decoys among strong BA-predicted binders.  
- **Training Data**: 399,392 entries (297,548 unique peptides, 44% hits) from monoallelic MS datasets (Sarkizova et al., Abelin et al.); includes top 2% strongest binders (hits and decoys) per MHCflurry BA predictions.  
- **Data Processing**: Peptides (8-11 mers) from source proteins; “with flanks” variant includes 5 upstream and downstream residues (25-mer total), “without flanks” uses peptide only (15-mer); padded with “X” and encoded with BLOSUM62.  
- **Encoding**: BLOSUM62 matrix converts each amino acid (and “X”) into a 21-dimensional vector for peptides and flanks (if used).  
- **MHC/Epitope Encoding**: N/A; designed to be allele-independent, so no MHC or epitope-specific encoding is applied.  
- **Architecture**: Ensemble of 8 CNNs (from 128 candidates); initial convolutional layer (11-17 kernel, 256-512 filters), followed by parallel 1D CNNs for N- and C-terminal cut-site predictions, integrated by a dense layer considering termini cleavability and flank properties.  
- **Loss Functions**: Binary cross-entropy loss, optimized with Adam and early stopping, comparing MS hits (1) to decoys (0).

## MHCflurry 2.0 PS
- **Purpose**:  
  - **What it does**: Combines BA and AP predictions to estimate the overall likelihood a peptide is presented on MHC class I.  
  - **Focus**: Integrated prediction of presentation, capturing both binding and processing effects.  
  - **Output**: A presentation score (0-1) reflecting the probability of MHC surface presentation.  
- **Name of the model**: MHCflurry 2.0 PS (presentation score model integrating BA and AP predictions, with “with flanks” and “without flanks” variants).  
- **Output of the model**: A score (0-1) representing the probability a peptide is presented on MHC class I, combining the strongest BA prediction across alleles and the AP prediction.  
- **Training Data**: 75,378 entries (24,983 hits) from 56 multiallelic MS samples (MULTIALLELIC-OLD); includes hits and 2x decoys sampled from the same proteins.  
- **Data Processing**: Takes two inputs: transformed BA prediction (tightest affinity across alleles, 0-1 scale) and AP prediction (with or without flanks); no additional encoding applied.  
- **Encoding**: N/A; relies on BLOSUM62 encodings from BA (peptides, MHC) and AP (peptides, flanks) predictors.  
- **MHC/Epitope Encoding**: N/A; no direct MHC/epitope encoding, as it integrates precomputed BA and AP scores.  
- **Architecture**: Simple logistic regression with 3 parameters (coefficients for BA and AP inputs, intercept); implemented in scikit-learn with LBFGS solver.  
- **Loss Functions**: Logistic loss (binary cross-entropy), optimized to distinguish MS hits from decoys in multiallelic data.

## HLApollo
- **Purpose**:  
  - **What it does**: Predicts the probability a peptide is presented on MHC class I, integrating binding and processing.  
  - **Focus**: Pan-allelic peptide-MHC-I presentation for personalized cancer vaccines.  
  - **Output**: A probability score (0-1) indicating likelihood of MHC-I presentation.  
- **Name of the model**: HLApollo (transformer-based, pan-allelic model for peptide-MHC-I presentation prediction).  
- **Output of the model**: Outputs a probability score (0-1) representing the likelihood a peptide is presented on MHC class I, derived from peptide, MHC, and flanking sequences; evaluated with average precision (AP) on skewed datasets (1:99 positives:negatives).  
- **Training Data**: 953,693 unique [peptide, genotype] tuples from 22 published studies and IEDB, covering 305,646 unique peptides, 171 HLA-I alleles, and 347 genotypes (single-allelic and multi-allelic data).  
- **Data Processing**: Inputs include peptide sequences (8-14 mers), MHC-I pseudosequences, and N- and C-terminal flanking sequences; processed end-to-end with multi-allelic deconvolution and negative set switching (new negative sets sampled each epoch).  
- **Encoding**: Learned embeddings for peptides, MHC pseudosequences, and flanks, generated by the transformer architecture; no predefined encoding like BLOSUM62 mentioned.  
- **MHC/Epitope Encoding**: Learned embeddings for peptides and MHC-I pseudosequences (processed separately then concatenated), capturing allele-specific and peptide-specific features.  
- **Architecture**: Transformer-based model; processes peptide and MHC pseudosequences separately with transformer encoder layers, concatenates representations, followed by 3 fully connected layers; trained with negative set switching for regularization.  
- **Loss Functions**: Cross-entropy loss, with single-allelic samples upweighted by a factor of 4, optimized using Adam with learning rates of 0.0002 (transformer) and 0.0004 (final layers).  

## NetMHCpan-4.1
- **Purpose**:  
  - **What it does**: Predicts the likelihood of peptide binding to any MHC class I molecule of known sequence.  
  - **Focus**: Pan-allelic MHC-I binding affinity and eluted ligand prediction for epitope identification.  
  - **Output**: A likelihood score (0-1) of a peptide being a natural MHC-I ligand, with optional affinity (nM).  
- **Name of the model**: NetMHCpan-4.1 (pan-specific MHC class I binding predictor using artificial neural networks).  
- **Output of the model**: Outputs a likelihood score (0-1) indicating the probability a peptide is a natural MHC class I ligand, with optional binding affinity in nanomolar (nM) units; evaluated using %Rank and binding thresholds (e.g., strong binder less than 0.5%, weak binder  less than 2%).  
- **Training Data**: Over 850,000 quantitative binding affinity (BA) and mass spectrometry eluted ligand (EL) peptides; BA covers 170+ MHC molecules (human HLA-A/B/C/E, mouse H-2, cattle BoLA, etc.), EL covers 177+ molecules, including single-allelic (SA) and multi-allelic (MA) data.  
- **Data Processing**: Inputs are peptide sequences (any length, typically 8-11 mers) and MHC protein sequences (or pseudosequences); multi-allelic data deconvolved using NNAlign_MA for pseudo-labeling; no explicit preprocessing like padding detailed on the site.  
- **Encoding**: Combines sparse encoding (one-hot-like) and BLOSUM matrix encoding for peptides; MHC encoded as pseudosequences (key binding residues).  
- **MHC/Epitope Encoding**: MHC encoded as pseudosequences (typically 34-37 residues from binding regions), processed with neural networks; peptides encoded with sparse and BLOSUM representations.  
- **Architecture**: Artificial neural network (ANN) ensemble, upgraded with NNAlign_MA for multi-allelic deconvolution; processes peptide and MHC inputs concurrently, followed by output layers for binding likelihood and affinity.  
- **Loss Functions**: Mean squared error (MSE) for binding affinity (BA) predictions, cross-entropy for eluted ligand (EL) likelihood; single-allelic samples upweighted in training.  

## MixMHCpred-3.0
- **Purpose**:  
  - **What it does**: Predicts peptides likely to be presented as HLA-I ligands on the cell surface.  
  - **Focus**: Multi-specificity modeling of MHC-I binding for epitope prioritization.  
  - **Output**: A binding score indicating peptide-HLA-I ligand likelihood, with %Rank for comparison.  
- **Name of the model**: MixMHCpred-3.0 (HLA-I ligand predictor using position weight matrices and motif deconvolution).  
- **Output of the model**: Outputs a raw binding score (higher is better, no upper limit) and %Rank (percentile vs. random peptides) for peptides binding to specified HLA-I alleles; designed for peptides 8-14 amino acids long, prioritizing likely ligands.  
- **Training Data**: Over 350,000 HLA-I ligands from mass spectrometry datasets (e.g., 50+ peptidomics datasets from Bassani-Sternberg et al., 2017, and mono-allelic cell lines), covering 58+ alleles; expanded in later versions (e.g., 2.2, 3.0) for broader coverage.  
- **Data Processing**: Inputs are peptide sequences (8-14 mers) in text or FASTA format; no explicit preprocessing like padding, but requires pre-aligned peptides for motif deconvolution; uses multi-specificity motifs per allele.  
- **Encoding**: Position Weight Matrices (PWMs) derived from MS data, adjusted for peptide length and cysteine bias; no neural network embeddings, relies on frequency-based scoring.  
- **MHC/Epitope Encoding**: N/A; does not explicitly encode MHC or epitopes beyond allele-specific PWMs trained on ligand data; alleles specified as input (e.g., A0101).  
- **Architecture**: Scoring algorithm based on Position Weight Matrixs (PWMs); uses unsupervised motif deconvolution (MixMHCp) to identify multiple binding motifs per allele, followed by log-likelihood scoring; no neural network, distinct from ANN-based tools.  
- **Loss Functions**: Log-likelihood scoring (implicit in PWM optimization), comparing peptide sequences to trained motifs; not a traditional loss function as it’s not a neural network, but optimized for motif fit.  

## HLAthena (MSiCEB)
- **Purpose**:  
  - **What it does**: Predicts the likelihood a peptide is presented on HLA class I, integrating binding and processing features.  
  - **Focus**: Accurate endogenous peptide presentation across diverse HLA alleles for immunotherapy.  
  - **Output**: A probability score (0-1) indicating HLA-I presentation likelihood, optimized for PPV.  
- **Name of the model**: HLAthena (MSiCEB) (integrative predictor for HLA class I peptide presentation, combining neural networks and logistic regression).  
- **Output of the model**: Outputs a probability score (0-1) representing the likelihood a peptide is presented on HLA class I, averaged across 3 runs of 5-fold cross-validation; evaluated with positive predictive value (PPV) at top 0.1% of 1:999 hit:decoy datasets.  
- **Training Data**: 186,464 HLA-I peptides eluted from 95 mono-allelic cell lines (31 HLA-A, 40 HLA-B, 21 HLA-C, 3 HLA-G), covering 8-11 mers, identified by LC-MS/MS from B721.221 cells.  
- **Data Processing**: Inputs are peptide sequences (8-11 mers) encoded with multiple methods; extrinsic features (cleavability, RNA-seq expression, gene presentation bias) integrated via logistic regression; no explicit padding, uses 10x hits and decoys for balanced training.  
- **Encoding**: Peptide sequences encoded with one-hot (binary), BLOSUM62, and PMBEC matrices; additional amino acid properties (3 PCs) and 8 peptide-level features (e.g., hydrophobicity) included in neural network input.  
- **MHC/Epitope Encoding**: N/A; allele-specific models use peptide features only, no explicit MHC encoding (pan-allele version adds MHC pocket features, but MSiCEB is allele-specific).  
- **Architecture**: Two-stage model: (1) Neural network (MSi) with one hidden layer (50 units, tanh activation) predicts intrinsic binding from peptide features; (2) Logistic regression (MSiCEB) integrates MSi scores with cleavability, expression, and gene presentation bias; trained with 5-fold CV and 3 random seeds.  
- **Loss Functions**: Cross-entropy loss for both neural network (MSi) and logistic regression (MSiCEB) stages, optimized over 10 epochs with early stopping on $20\%$ hold-out data.  

## TransPHLA
- **Purpose**:  
  - **What it does**: Predicts the binding affinity between peptides and HLA class I alleles for vaccine design.  
  - **Focus**: Pan-specific pHLA binding to identify neoantigens and optimize peptides.  
  - **Output**: A binding score (0-1) indicating peptide-HLA-I binding likelihood.  
- **Name of the model**: TransPHLA (transformer-based model for peptide-HLA class I binding prediction within the TransMut framework).  
- **Output of the model**: Outputs a binding score (0-1) representing the likelihood of peptide-HLA-I binding, evaluated via AUC, accuracy, MCC, and F1; achieves 96% neoantigen screening rate and outperforms 14 benchmarks.  
- **Training Data**: Over 170,000 pHLA pairs (8-14 mers) from IEDB and other sources, including independent (112 HLA alleles) and external (5 HLA alleles) test sets, plus 221 neoantigen and 278 HPV vaccine samples for validation.  
- **Data Processing**: Inputs are peptide sequences (8-14 mers) and full HLA-I sequences; padded positions masked in transformer self-attention; no explicit preprocessing like flanking sequences mentioned.  
- **Encoding**: Learned embeddings for peptides and HLA sequences, with positional embeddings added to capture sequence position information, processed by transformer’s embedding block.  
- **MHC/Epitope Encoding**: Learned embeddings for both peptides and HLA sequences, jointly processed via self-attention to model pHLA interactions; no predefined encoding (e.g., BLOSUM) specified.  
- **Architecture**: Transformer model with four sub-modules: (1) embedding block (sequence + positional embeddings), (2) encoder block (multi-head self-attention with padding masks), (3) feature optimization block (fully connected layers with gyro channel), (4) projection block (fully connected layers for binding score); trained on GPU (28s for 170K predictions).  
- **Loss Functions**: Cross-entropy loss for binary classification (binding vs. non-binding), optimized to distinguish binders (e.g., IC50 < 500 nM) from non-binders.  

In [9]:
import pandas as pd

# Making columns to store attributes for different models
columns = [
    "Model",
    "Output",
    "Training Data",
    "Data Processing",
    "Encoding",
    "MHC/Epitope Encoding",
    "Architecture",
    "Loss Functions",
    "Notes",
    "Purpose"
]


data = [
    [
        "ESM-2",
        "Masked AA probs (15% of seq)",
        "~65M UniRef50 seqs",
        "15% random masking",
        "Learned embeddings",
        "N/A",
        "Transformer",
        "Masked LM loss",
        "Scales to 15B params",
        "Sequence pattern learning"
    ],
    [
        "ESMFold",
        "3D coords, pLDDT, pTM",
        "~325K PDB + 12M AF2 structs",
        "ESM-2 reps + PDB/AF2 data",
        "Learned embeddings",
        "N/A",
        "Transformer + folding trunk",
        "AlphaFold-style losses",
        "60x faster than AF2",
        "Structure prediction"
    ],
    [
        "MHCflurry 2.0 BA",
        "Binding affinity (0-1)",
        "493K MS + 219K affinity",
        "45-mer padded, BLOSUM62",
        "BLOSUM62",
        "BLOSUM62 (peptides, MHC)",
        "Feedforward NN ensemble",
        "MSE variant",
        "Pan-allele, 14K alleles",
        "MHC binding affinity"
    ],
    [
        "MHCflurry 2.0 AP",
        "Processing prob (0-1)",
        "399K monoallelic MS peptides",
        "15/25-mer padded, BLOSUM62",
        "BLOSUM62",
        "N/A",
        "CNN ensemble",
        "Binary cross-entropy",
        "Allele-independent",
        "Antigen processing"
    ],
    [
        "MHCflurry 2.0 PS",
        "Presentation score (0-1)",
        "75K multiallelic MS entries",
        "BA + AP inputs",
        "N/A",
        "N/A",
        "Logistic regression",
        "Logistic loss",
        "Combines BA & AP",
        "Presentation prediction"
    ]
  ,
    [
    "HLApollo",
    "Presentation prob (0-1)",
    "953K pMHC-I tuples",
    "Peptide + MHC + flanks",
    "Learned embeddings",
    "Learned embeddings (peptide, MHC)",
    "Transformer",
    "Cross-entropy loss",
    "Pan-allelic, 12.65% AP boost",
    "MHC-I presentation"
],

    [
    "NetMHCpan-4.1",
    "Binding likelihood (0-1)",
    ">850K BA + EL peptides",
    "Peptide + MHC seqs",
    "Sparse + BLOSUM",
    "Pseudo-seq (MHC)",
    "Neural network (ANN)",
    "MSE + cross-entropy",
    "Pan-allelic, MA deconvolution",
    "MHC-I binding prediction"
],

    [
    "MixMHCpred-3.0",
    "Binding score",
    ">350K HLA-I ligands",
    "Peptide seqs (8-14)",
    "PWM-based",
    "N/A",
    "Scoring algorithm",
    "Log-likelihood",
    "Multi-specificity",
    "MHC-I ligand prediction"
],

    [
    "HLAthena (MSiCEB)",
    "Presentation prob (0-1)",
    "186K HLA-I peptides",
    "Peptide seqs + extrinsic",
    "One-hot + BLOSUM + PMBEC",
    "N/A",
    "NN + logistic regression",
    "Cross-entropy",
    "1.5x PPV boost",
    "MHC-I presentation"
],
    [
    "TransPHLA",
    "Binding score (0-1)",
    ">170K pHLA pairs",
    "Peptide + HLA seqs",
    "Learned embeddings",
    "Learned embeddings (peptide, HLA)",
    "Transformer",
    "Cross-entropy",
    "Pan-specific, 96% neoantigen rate",
    "pHLA binding prediction"
]

]



df = pd.DataFrame(data, columns=columns)
print(df.shape)

df

(10, 10)


,Model,Output,Training Data,Data Processing,Encoding,MHC/Epitope Encoding,Architecture,Loss Functions,Notes,Purpose
0,ESM-2,Masked AA probs (15% of seq),~65M UniRef50 seqs,15% random masking,Learned embeddings,N/A,Transformer,Masked LM loss,Scales to 15B params,Sequence pattern learning
1,ESMFold,"3D coords, pLDDT, pTM",~325K PDB + 12M AF2 structs,ESM-2 reps + PDB/AF2 data,Learned embeddings,N/A,Transformer + folding trunk,AlphaFold-style losses,60x faster than AF2,Structure prediction
2,MHCflurry 2.0 BA,Binding affinity (0-1),493K MS + 219K affinity,"45-mer padded, BLOSUM62",BLOSUM62,"BLOSUM62 (peptides, MHC)",Feedforward NN ensemble,MSE variant,"Pan-allele, 14K alleles",MHC binding affinity
3,MHCflurry 2.0 AP,Processing prob (0-1),399K monoallelic MS peptides,"15/25-mer padded, BLOSUM62",BLOSUM62,N/A,CNN ensemble,Binary cross-entropy,Allele-independent,Antigen processing
4,MHCflurry 2.0 PS,Presentation score (0-1),75K multiallelic MS entries,BA + AP inputs,N/A,N/A,Logistic regression,Logistic loss,Combines BA & AP,Presentation prediction
5,HLApollo,Presentation prob (0-1),953K pMHC-I tuples,Peptide + MHC + flanks,Learned embeddings,"Learned embeddings (peptide, MHC)",Transformer,Cross-entropy loss,"Pan-allelic, 12.65% AP boost",MHC-I presentation
6,NetMHCpan-4.1,Binding likelihood (0-1),>850K BA + EL peptides,Peptide + MHC seqs,Sparse + BLOSUM,Pseudo-seq (MHC),Neural network (ANN),MSE + cross-entropy,"Pan-allelic, MA deconvolution",MHC-I binding prediction
7,MixMHCpred-3.0,Binding score,>350K HLA-I ligands,Peptide seqs (8-14),PWM-based,N/A,Scoring algorithm,Log-likelihood,Multi-specificity,MHC-I ligand prediction
8,HLAthena (MSiCEB),Presentation prob (0-1),186K HLA-I peptides,Peptide seqs + extrinsic,One-hot + BLOSUM + PMBEC,N/A,NN + logistic regression,Cross-entropy,1.5x PPV boost,MHC-I presentation
9,TransPHLA,Binding score (0-1),>170K pHLA pairs,Peptide + HLA seqs,Learned embeddings,"Learned embeddings (peptide, HLA)",Transformer,Cross-entropy,"Pan-specific, 96% neoantigen rate",pHLA binding prediction
